In [1]:
import pandas as pd

df = pd.read_parquet("../../data/HI-Medium_Trans.parquet")
print(df.shape)

(31898238, 11)


In [2]:
print("Date range:", df["Timestamp"].min(), "to", df["Timestamp"].max())
print()
print("Missing values per column:")
print(df.isna().sum())
print()
print("Payment formats:")
print(df["Payment Format"].value_counts())
print()
print("Top 10 payment currencies:")
print(df["Payment Currency"].value_counts().head(10))
print()
print("Amount summary:")
print(df[["Amount Received", "Amount Paid"]].describe().round(2))

Date range: 2022-09-01 00:00:00 to 2022-09-28 15:58:00

Missing values per column:
Timestamp             0
From Bank             0
from_account          0
To Bank               0
to_account            0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
Is Laundering         0
dtype: int64

Payment formats:
Payment Format
Cheque          12280058
Credit Card      8777816
ACH              3868410
Cash             3217531
Reinvestment     1945611
Wire             1119774
Bitcoin           689038
Name: count, dtype: int64

Top 10 payment currencies:
Payment Currency
US Dollar            11688249
Euro                  7315958
Yuan                  2330638
Shekel                1408707
Canadian Dollar       1078358
UK Pound              1018575
Ruble                  975525
Australian Dollar      918569
Yen                    856132
Swiss Franc            844623
Name: count, dtype: int64

Amount summary:
       Amount Rece

In [3]:
daily = df.groupby(df["Timestamp"].dt.date)["Is Laundering"].agg(["count", "sum"])
daily.columns = ["transactions", "laundering"]
print(daily)
print()

# Make both currency columns use the same category list so we can compare them
pay = df["Payment Currency"]
rec = df["Receiving Currency"].cat.set_categories(pay.cat.categories)
print("Share of cross-currency transactions:", round((pay != rec).mean() * 100, 2), "%")


            transactions  laundering
Timestamp                           
2022-09-01       4465985        1056
2022-09-02       3021258        1331
2022-09-03        828958        1373
2022-09-04        830607        1549
2022-09-05       1928599        1863
2022-09-06       1929090        2041
2022-09-07       1931015        2147
2022-09-08       1930610        2154
2022-09-09       2621004        2199
2022-09-10        831874        2081
2022-09-11        830143        1966
2022-09-12       1930240        2179
2022-09-13       1928626        2275
2022-09-14       1930957        2263
2022-09-15       1930419        2253
2022-09-16       3021866        2416
2022-09-17          2020        1199
2022-09-18          1591         916
2022-09-19          1195         695
2022-09-20           855         492
2022-09-21           586         336
2022-09-22           309         184
2022-09-23           134          84
2022-09-24           104          65
2022-09-25           102          59
2

In [4]:
df["cross_currency"] = (pay != rec)

print("Laundering rate by payment format (per 10,000):")
print((df.groupby("Payment Format", observed=True)["Is Laundering"].mean() * 10000).round(2))
print()
print("Laundering rate by cross-currency (per 10,000):")
print((df.groupby("cross_currency")["Is Laundering"].mean() * 10000).round(2))

Laundering rate by payment format (per 10,000):
Payment Format
ACH             79.48
Bitcoin          3.54
Cash             2.07
Cheque           1.81
Credit Card      1.54
Reinvestment     0.00
Wire             0.00
Name: Is Laundering, dtype: float64

Laundering rate by cross-currency (per 10,000):
cross_currency
False    11.22
True      0.00
Name: Is Laundering, dtype: float64


In [5]:
cutoff = pd.Timestamp("2022-09-17")
before_rows = len(df)
before_laundering = int(df["Is Laundering"].sum())

df = df[df["Timestamp"] < cutoff].copy()

print("Rows removed:", before_rows - len(df))
print("Laundering rows removed:", before_laundering - int(df["Is Laundering"].sum()))
print("Rows left:", len(df))
print("Laundering rows left:", int(df["Is Laundering"].sum()))
print("New date range:", df["Timestamp"].min(), "to", df["Timestamp"].max())

Rows removed: 6987
Laundering rows removed: 4084
Rows left: 31891251
Laundering rows left: 31146
New date range: 2022-09-01 00:00:00 to 2022-09-16 23:59:00
